In [0]:
# ==========================================================
# 05_silver_meta_municipio
# Camada: Silver
# Fonte: 2026-11-01_meta_alfabetizacao_municipio.parquet (Bronze)
# Destino: default.meta_municipio
#
# RESPONSABILIDADE
# Transformar a publicação oficial das metas de alfabetização
# (nível município) em tabela Silver confiável.
#
# NOTA SOBRE A COLUNA rede
# Nesta tabela rede é texto ('Municipal'), não código numérico
# como no indicador. O join com indicador_municipio é por
# id_municipio e ano, não por rede.
#
# NOTA SOBRE nivel_alfabetizacao
# Coluna exclusiva desta tabela. Preservada por ser informação
# que não existe nas demais fontes.
#
# NOTA DE PERMISSÃO
# O schema silver pertence a outro usuário. A gravação ocorre
# em default, conforme adaptação documentada na seção 10.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "meta_municipio"

BASE_BRONZE = (
    "/Workspace/Users/rm374983@fiap.com.br/"
    "postech-2iast-fase3-modelo-preditivo-g28/dados/bronze"
)
ARQUIVO_ORIGEM = "2026-11-01_meta_alfabetizacao_municipio.parquet"
CAMINHO_ORIGEM = f"{BASE_BRONZE}/{ARQUIVO_ORIGEM}"

TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {CAMINHO_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
# ==========================================================
# Célula 2 — Leitura da Bronze
# ==========================================================

df_raw = spark.read.parquet(CAMINHO_ORIGEM)

print(f"Registros lidos: {df_raw.count():,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Transformação e tipagem
#
# Decisões:
# - nivel_alfabetizacao é mantida: coluna exclusiva desta
#   tabela, sem equivalente nas demais fontes.
# - taxa_alfabetizacao e percentual_participacao mantidas.
# - As 7 colunas de meta anual são mantidas como estão.
# - Dedupe por (ano, id_municipio, rede).
# ==========================================================

df_silver = (
    df_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("id_municipio").cast("string").alias("id_municipio"),
        F.col("rede").cast("string").alias("rede"),
        F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
        *[
            F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_alfabetizacao_{a}")
            for a in range(2024, 2031)
        ],
        F.col("nivel_alfabetizacao").cast("int").alias("nivel_alfabetizacao"),
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
    )
    .filter(F.col("ano").isNotNull() & F.col("id_municipio").isNotNull())
    .dropDuplicates(["ano", "id_municipio", "rede"])
)

print(f"Registros após transformação: {df_silver.count():,}")

In [0]:
# ==========================================================
# Célula 4 — Rastreabilidade
# ingested_at, source e version em todos os registros.
# ==========================================================

df_silver = (
    df_silver
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit(ARQUIVO_ORIGEM))
    .withColumn("version", F.lit("v2"))
)

df_silver.select(
    "ano", "id_municipio", "rede", "taxa_alfabetizacao",
    "meta_alfabetizacao_2024", "nivel_alfabetizacao",
    "ingested_at", "source", "version"
).show(5, truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Data Quality
# Regras: completude, domínio e unicidade.
#
# ATENÇÃO: taxa nula NÃO é falha crítica aqui. Municípios
# sem avaliação no ano têm linha com valores nulos na origem.
# A regra crítica é sobre a chave (ano, id_municipio, rede).
# ==========================================================

total = df_silver.count()

nulos_id = df_silver.filter(F.col("id_municipio").isNull()).count()
nulos_rede = df_silver.filter(F.col("rede").isNull()).count()

# Domínio: código IBGE de município tem 7 dígitos
ids_invalidos = df_silver.filter(
    F.length(F.col("id_municipio")) != 7
).count()

# Domínio: metas preenchidas devem estar entre 0 e 100
metas = [f"meta_alfabetizacao_{a}" for a in range(2024, 2031)]
fora_faixa = df_silver.filter(
    F.greatest(*[
        F.when((F.col(m) < 0) | (F.col(m) > 100), 1).otherwise(0) for m in metas
    ]) == 1
).count()

chave = ["ano", "id_municipio", "rede"]
duplicatas = total - df_silver.dropDuplicates(chave).count()

# Diagnóstico: linhas sem taxa (municípios sem avaliação no ano)
sem_taxa = df_silver.filter(F.col("taxa_alfabetizacao").isNull()).count()

print("=" * 55)
print("DATA QUALITY — meta_municipio")
print("=" * 55)
print(f"Registros totais     : {total:,}")
print(f"ID município nulo    : {nulos_id:,}")
print(f"Rede nula            : {nulos_rede:,}")
print(f"ID fora do padrão    : {ids_invalidos:,}")
print(f"Meta fora de 0-100   : {fora_faixa:,}")
print(f"Duplicatas na chave  : {duplicatas:,}")
print(f"Linhas sem taxa      : {sem_taxa:,} ({sem_taxa/total*100:.2f}%)")
print(f"Anos                 : {[r[0] for r in df_silver.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Municípios distintos : {df_silver.select('id_municipio').distinct().count():,}")
print("=" * 55)

assert nulos_id == 0, "FALHA CRÍTICA: id_municipio nulo"
assert nulos_rede == 0, "FALHA CRÍTICA: rede nula"
assert ids_invalidos == 0, "FALHA CRÍTICA: id_municipio fora do padrão de 7 dígitos"
assert fora_faixa == 0, "FALHA CRÍTICA: meta fora do domínio 0-100"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação da Silver
# ==========================================================

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
# ==========================================================
# Célula 7 — Validação pós-gravação
# ==========================================================

df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Municípios          : {df_check.select('id_municipio').distinct().count():,}")

print("\nCobertura por ano:")
df_check.groupBy("ano").agg(
    F.count("*").alias("linhas"),
    F.count("taxa_alfabetizacao").alias("com_taxa"),
    F.count("meta_alfabetizacao_2024").alias("com_meta_2024"),
).orderBy("ano").show()

print("\nDistribuição de nivel_alfabetizacao:")
df_check.groupBy("nivel_alfabetizacao").count().orderBy("nivel_alfabetizacao").show()

print("\nAmostra:")
df_check.select(
    "ano", "id_municipio", "taxa_alfabetizacao",
    "meta_alfabetizacao_2024", "meta_alfabetizacao_2025",
    "nivel_alfabetizacao", "percentual_participacao"
).orderBy("id_municipio", "ano").show(10, truncate=False)